In [1]:
import boto3
import json
import pandas as pd

# === Load Economic Data ===
def load_economic_data(filepath):
    df = pd.read_csv(filepath)
    df["Date"] = pd.to_datetime(df["Date"])  # Ensure Date is in datetime format
    return df

# === Load Daily Fed Funds Target Rate Data ===
def load_fed_funds_data(filepath):
    df = pd.read_csv(filepath)
    df["Date"] = pd.to_datetime(df["Date"])  # Ensure Date is in datetime format
    return df


In [2]:
def format_prompt(data, target_date, n, template):
    """
    Formats the Claude prompt using the provided data.

    Parameters:
        data (DataFrame): The n months of economic data
        target_date (datetime): The prediction date
        n (int): Number of months used
        template (str): Optional custom prompt template with placeholders

    Returns:
        str: Full prompt string
    """
    data_table = data.to_markdown(index=False)

    if template is None:
        # Default fallback template
        template = f"""
You are an AI economist. Based on the past {n} months of U.S. macroeconomic data,
predict the Federal Funds Target Rate for {target_date.date()}.  

Respond in this JSON format:
{{
  "predicted_rate": "e.g. 5.25",
      "confidence": "e.g. 85%",
      "explanation": "Brief rationale, max 100 words"
}}

{data_table}
"""
    else:
        template = template.format(n=n, target_date=str(target_date), data_table=data_table)

    return template


In [3]:
def query_claude_aws(prompt, model_id="anthropic.claude-3-5-sonnet-20240620-v1:0", max_tokens=500):
    bedrock = boto3.client("bedrock-runtime")

    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "temperature": 0.5,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ]
    }

    response = bedrock.invoke_model(
        modelId=model_id,
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json"
    )

    response_body = json.loads(response["body"].read())
    message_content = response_body["content"][0]["text"]

    try:
        # Try to parse the returned JSON text
        parsed_response = json.loads(message_content)
        return parsed_response
    except json.JSONDecodeError:
        # If parsing fails, return raw text in a fallback format
        return {
            "predicted_rate": "Unknown",
            "confidence": "Unknown",
            "explanation": message_content[:200] + "..."  # truncate long response
        }


In [28]:
def run_sensitivity_analysis_v2(economic_df, target_date, n, shocks):
    """
    Run sensitivity analysis by shocking one or multiple variables.

    Parameters:
        economic_df (DataFrame): Historical macroeconomic data.
        target_date (Timestamp): Date for which to predict Fed decision.
        n (int): Number of months of data to include.
        shocks (dict): Dictionary where keys are variable names (str) and values are shock amounts (float).

    Returns:
        dict: Original and shocked scenario results.
    """
    past_data = economic_df[economic_df["Date"] < target_date].sort_values("Date", ascending=False)
    base_data = past_data.head(n)

    shocked_data = base_data.copy()
    for var, shock in shocks.items():
        if var in shocked_data.columns:
            # shocked_data[var] += shock
            shocked_data[var] *= (1 + shock / 100)

        else:
            raise ValueError(f"Column '{var}' not found in economic data.")

    prompt_base = format_prompt(base_data, target_date, n, template=None)
    prompt_shocked = format_prompt(shocked_data, target_date, n, template=None)

    response_base = query_claude_aws(prompt_base)
    response_shocked = query_claude_aws(prompt_shocked)

    shock_desc = ', '.join([f"{k} +{v}" for k, v in shocks.items()])
    return {
        "Base": response_base,
        f"Shocked ({shock_desc})": response_shocked
    }


In [32]:
# Shock unemployment by +2 and inflation by +1
shocks = {"U3": 2.0, "U6": 2.0, "CPI": 5.0}

sensitivity_results = run_sensitivity_analysis_v2(
    economic_df=economic_df,
    target_date=pd.Timestamp("2025-03-01"),
    n=6,
    shocks=shocks
)


In [33]:
print(json.dumps(sensitivity_results, indent=2))

{
  "Base": {
    "predicted_rate": "4.75",
    "confidence": "70%",
    "explanation": "Given stable inflation expectations, moderating CPI and PCE growth, and a slight uptick in unemployment, the Fed may cautiously lower rates. However, persistent wage growth and GDP expansion suggest a gradual approach. The predicted 4.75% balances inflation control with economic support."
  },
  "Shocked (U3 +2.0, U6 +2.0, CPI +5.0)": {
    "predicted_rate": "4.75",
    "confidence": "70%",
    "explanation": "Based on recent trends, inflation is moderating but still above target. Labor market remains tight with low unemployment. GDP growth is slowing but positive. The Fed may cautiously lower rates from current levels, balancing inflation concerns against economic growth support. However, uncertainty remains high due to potential economic shifts."
  }
}
